[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-01-ray-core-tasks.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Ray Core — Remote Functions and the Task Model
**certified-journeys / ray-certified** · Day 1 · Foundations

> **Goal for today:** Understand Ray's task model — how `@ray.remote` turns ordinary Python functions into distributed tasks, how `ObjectRef` represents a future, and how `ray.get()` collects results without blocking parallelism.


In [ ]:
%pip install -q 'ray[default]'


## Step 1 · What Is Ray?

Ray is an open-source distributed computing framework that lets you **scale Python code from a laptop to a cluster** with minimal changes. Its core abstraction is the **task** — a remote function that runs asynchronously, possibly on a different machine.

| Feature | Ray | Celery | Dask |
|---------|-----|--------|------|
| Primary abstraction | Tasks + Actors | Tasks (queue-based) | DataFrames + Delayed |
| Stateful workers | ✅ Actors | ⚠ Limited | ❌ |
| Shared memory | ✅ Plasma store | ❌ | ⚠ Partial |
| ML ecosystem | ✅ Ray Train/Tune/Serve | ❌ | ⚠ Partial |
| Local cluster | ✅ Auto | Requires broker | ✅ Auto |

Ray's key insight: **hide distributed systems complexity behind familiar Python syntax**. A function decorated with `@ray.remote` behaves like any Python function — except it returns an `ObjectRef` (a future) instead of the result directly.


In [ ]:
import ray
import time

# Initialize Ray — without arguments it starts a local cluster automatically.
# On Colab this spins up a single-node cluster using all available CPUs.
ray.init(ignore_reinit_error=True)

print(f"Ray version      : {ray.__version__}")
print(f"Available CPUs   : {ray.available_resources().get('CPU', 0):.0f}")
print(f"Available memory : {ray.available_resources().get('memory', 0) / 1e9:.2f} GB")


### What just happened?
- `ray.init()` started a **local Ray cluster** — a head node with a Global Control Store (GCS) and a Plasma object store.
- `ignore_reinit_error=True` lets us re-run the cell safely without crashing if Ray is already running.
- **Available CPUs** is the parallelism budget Ray will use to schedule tasks on this machine.
- No configuration file is needed for local development — `ray.init()` handles everything automatically.


## Step 2 · Decorating a Function with `@ray.remote`

Adding `@ray.remote` to a Python function does two things:
1. **Registers** the function with the Ray scheduler.
2. **Replaces** the function object with a `RemoteFunction` — calling it now requires `.remote()`.

```
Normal call :  result   = my_fn(x)           # blocks until done
Ray call    :  ref      = my_fn.remote(x)    # returns ObjectRef immediately
              result   = ray.get(ref)        # blocks only here
```

The `ObjectRef` is a **handle to a future result** stored in Ray's distributed object store. It can be passed to other tasks as an argument — Ray will automatically resolve it before the task starts.


In [ ]:
# Define a plain Python function — nothing special yet
def slow_square(x: int) -> int:
    """Simulate a CPU-bound computation by sleeping."""
    time.sleep(1)   # pretend this takes 1 second
    return x * x

# Now make it a Ray remote function with a single decorator
@ray.remote
def slow_square_remote(x: int) -> int:
    """Same computation, but runs in a Ray worker process."""
    time.sleep(1)
    return x * x

# --- Sequential (no Ray) ---
start = time.time()
results_seq = [slow_square(i) for i in range(4)]
seq_time = time.time() - start
print(f"Sequential : {results_seq}  ({seq_time:.2f}s)")

# --- Parallel (with Ray) ---
# Submit ALL tasks first — each .remote() call returns immediately
start = time.time()
refs = [slow_square_remote.remote(i) for i in range(4)]   # non-blocking!
print(f"ObjectRefs : {refs}")                              # show the futures
results_par = ray.get(refs)                                 # block once at the end
par_time = time.time() - start
print(f"Parallel   : {results_par}  ({par_time:.2f}s)")
print(f"Speedup    : {seq_time / par_time:.1f}x")


### What just happened?
- **Each `.remote()` call returned immediately** — the tasks ran in parallel in worker processes.
- `refs` is a list of `ObjectRef` objects — they look like `ObjectRef(abc123...)` when printed.
- **`ray.get(refs)` blocked once** until all 4 results were ready, then returned them in order.
- The wall-clock time collapsed from ~4 s (sequential) to ~1 s (parallel) because all 4 tasks overlapped.


## Step 3 · `ray.get()` — Single vs List Patterns

There are two ways to call `ray.get()`:

| Pattern | Code | When to use |
|---------|------|-------------|
| Single ref | `result = ray.get(ref)` | You need exactly one result |
| List of refs | `results = ray.get([ref1, ref2, ...])` | You submitted a batch of tasks |

**Anti-pattern to avoid:**
```python
# BAD — interleaving .remote() and ray.get() serialises execution
for x in inputs:
    result = ray.get(slow_fn.remote(x))   # blocks after each submit!

# GOOD — submit all, then collect
refs    = [slow_fn.remote(x) for x in inputs]
results = ray.get(refs)
```


In [ ]:
@ray.remote
def compute(x: float) -> dict:
    """Return a small dict result — demonstrates non-trivial return types."""
    time.sleep(0.5)
    return {"input": x, "square": x**2, "cube": x**3}

# --- Pattern A: single ref ---
single_ref = compute.remote(5.0)
print(f"Type of ObjectRef: {type(single_ref).__name__}")
single_result = ray.get(single_ref)
print(f"Single result    : {single_result}")

# --- Pattern B: list of refs (correct batch pattern) ---
inputs = [1.0, 2.0, 3.0, 4.0, 5.0]
refs = [compute.remote(x) for x in inputs]   # all submitted at once
results = ray.get(refs)                        # one blocking call
print(f"\nBatch results ({len(results)} items):")
for r in results:
    print(f"  {r}")

# --- Anti-pattern demo (sequential despite using Ray) ---
print("\nAnti-pattern timing (interleaved get):")
start = time.time()
for x in inputs[:3]:
    r = ray.get(compute.remote(x))   # blocks each iteration — no parallelism!
print(f"  {time.time()-start:.2f}s  (should be ~1.5s, not ~0.5s)")

print("\nCorrect pattern timing:")
start = time.time()
results = ray.get([compute.remote(x) for x in inputs[:3]])
print(f"  {time.time()-start:.2f}s  (should be ~0.5s)")


### What just happened?
- **`ObjectRef` is serialisable** — you can pass it between tasks; Ray resolves it lazily.
- The anti-pattern forced the loop to wait ~0.5 s per iteration — equivalent to sequential execution.
- **The correct pattern** submits all tasks in one list comprehension, then calls `ray.get` exactly once.
- `ray.get` returns results **in submission order**, even though tasks may complete in any order.


## Step 4 · Passing ObjectRefs as Arguments — Task Pipelining

Ray tasks can accept `ObjectRef` objects as arguments. When Ray schedules the downstream task, it **automatically waits** for the upstream result without you calling `ray.get()`. This enables **task pipelining** — chaining tasks into a DAG.

```
ref_a = stage_a.remote(data)        # stage A runs
ref_b = stage_b.remote(ref_a)       # stage B starts as soon as A finishes
result = ray.get(ref_b)             # only block at the very end
```

This is how Ray builds computation graphs — the same pattern used by Ray Train and Ray Tune internally.


In [ ]:
@ray.remote
def extract(n: int) -> list:
    """Stage 1 — produce raw data."""
    time.sleep(0.3)
    return list(range(n))

@ray.remote
def transform(data: list) -> list:
    """Stage 2 — square every element."""
    time.sleep(0.3)
    return [x**2 for x in data]

@ray.remote
def aggregate(data: list) -> dict:
    """Stage 3 — summarise."""
    time.sleep(0.2)
    return {"count": len(data), "sum": sum(data), "max": max(data)}

# Build the pipeline — no ray.get() between stages!
# Each stage receives the ObjectRef from the previous stage.
ref_raw    = extract.remote(10)          # returns ObjectRef
ref_xfm    = transform.remote(ref_raw)   # passes ObjectRef directly — Ray resolves it
ref_result = aggregate.remote(ref_xfm)  # same pattern

# Only block once at the very end
final = ray.get(ref_result)
print(f"Pipeline result : {final}")

# Fan-out: run multiple pipelines concurrently
print("\nFan-out across 5 independent pipelines:")
pipeline_refs = []
for n in [5, 10, 15, 20, 25]:
    r1 = extract.remote(n)
    r2 = transform.remote(r1)
    r3 = aggregate.remote(r2)
    pipeline_refs.append(r3)

all_results = ray.get(pipeline_refs)
for n, res in zip([5, 10, 15, 20, 25], all_results):
    print(f"  n={n:<3} → {res}")


### What just happened?
- We built a **3-stage ETL pipeline** without any `ray.get()` between stages — Ray handles the dependency graph.
- **Fan-out** ran 5 independent pipelines concurrently by submitting all chains before calling `ray.get`.
- Ray's scheduler sees the `ObjectRef` arguments and automatically creates a task dependency — this is a DAG.
- **No explicit DAG definition required** — the data flow through `ObjectRef` arguments is the DAG.


## Step 5 · Resource Requests — `num_cpus`, `num_gpus`, Memory

By default, each Ray task requests **1 CPU**. You can override this with `@ray.remote(num_cpus=N)`. Ray uses these requests for **scheduling decisions** — it won't schedule a task if the cluster doesn't have enough free resources.

| Parameter | Effect | Example |
|-----------|--------|---------|
| `num_cpus` | CPU slots reserved | `@ray.remote(num_cpus=2)` |
| `num_gpus` | GPU units reserved | `@ray.remote(num_gpus=1)` |
| `memory` | Min RAM in bytes | `@ray.remote(memory=2*1024**3)` — 2 GB |
| `max_retries` | Auto-retry on failure | `@ray.remote(max_retries=3)` |

Use fractional CPUs (`num_cpus=0.5`) to pack more tasks per node for I/O-bound work.


In [ ]:
import os

# Task that requests 2 CPUs — Ray will only schedule it when 2 CPUs are free
@ray.remote(num_cpus=2)
def cpu_heavy_task(task_id: int) -> dict:
    """Simulate a task that benefits from 2 CPU threads."""
    time.sleep(0.5)
    return {
        "task_id": task_id,
        "worker_pid": os.getpid(),   # shows that tasks run in separate processes
        "status": "done"
    }

# Fractional CPU — useful for I/O-bound tasks (network calls, DB queries)
@ray.remote(num_cpus=0.25)
def io_bound_task(task_id: int) -> dict:
    """I/O-bound task — uses 1/4 CPU slot so many can run simultaneously."""
    time.sleep(0.3)   # simulate network wait
    return {"task_id": task_id, "worker_pid": os.getpid()}

# Check available CPUs on this cluster
total_cpus = ray.available_resources().get('CPU', 1)
print(f"Total available CPUs: {total_cpus}")

# Run CPU-heavy tasks — each consumes 2 CPUs so they run in pairs
print("\nRunning 4 cpu_heavy_task (2 CPUs each):")
refs = [cpu_heavy_task.remote(i) for i in range(4)]
for r in ray.get(refs):
    print(f"  {r}")

# Run many I/O-bound tasks — 0.25 CPU each, so up to 4x more concurrency
print("\nRunning 8 io_bound_task (0.25 CPU each):")
refs = [io_bound_task.remote(i) for i in range(8)]
pids = {r['worker_pid'] for r in ray.get(refs)}
print(f"  Completed on {len(pids)} unique worker PIDs")

# Demonstrate runtime resource override (.options())
print("\nUsing .options() to override resources at call time:")
ref = slow_square_remote.options(num_cpus=0.5).remote(7)
print(f"  Result: {ray.get(ref)}")


### What just happened?
- `@ray.remote(num_cpus=2)` made the scheduler hold 2 CPU slots before running each task.
- **`os.getpid()`** confirmed that tasks run in separate worker processes — true parallelism, no GIL.
- Fractional CPUs pack more concurrent tasks onto the same hardware for I/O-bound workloads.
- **`.options()`** lets you override decorator-level resources at call time without redefining the function.


## Step 6 · `ray.wait()` — Progress Streaming and Partial Results

`ray.get(refs)` waits for **all** results. `ray.wait(refs, num_returns=k)` returns as soon as `k` tasks are done — useful for processing results as they arrive (streaming pattern) or implementing timeouts.

```python
done, remaining = ray.wait(refs, num_returns=1, timeout=5.0)
# done      → list of completed ObjectRefs (up to num_returns)
# remaining → list of still-pending ObjectRefs
```


In [ ]:
import random

@ray.remote
def variable_latency_task(task_id: int) -> dict:
    """Each task sleeps a random amount — simulates real heterogeneous work."""
    latency = random.uniform(0.1, 0.8)
    time.sleep(latency)
    return {"task_id": task_id, "latency_s": round(latency, 3)}

random.seed(42)
refs = [variable_latency_task.remote(i) for i in range(8)]

print("Processing results as they complete (ray.wait streaming pattern):")
remaining = refs.copy()
order = []
while remaining:
    # Wait for the NEXT completed task (num_returns=1)
    done, remaining = ray.wait(remaining, num_returns=1)
    result = ray.get(done[0])    # done is a list with 1 element
    order.append(result['task_id'])
    print(f"  Completed task {result['task_id']}  latency={result['latency_s']}s")

print(f"\nCompletion order: {order}")
print("(Tasks finish in latency order, not submission order)")


### What just happened?
- **`ray.wait()` returned tasks as they finished** — shorter-latency tasks came back first.
- The `done / remaining` pattern is the standard Ray idiom for streaming result processing.
- This is essential for **best-of-N** patterns: cancel remaining tasks once you have enough results.
- Compare: `ray.get` would have forced you to wait for the slowest task before processing any result.


In [ ]:
# Challenge: Parallel word-frequency counter
#
# You have a list of text chunks. Write a @ray.remote function `count_words`
# that takes a string chunk and returns a dict mapping word -> count.
# Then:
#   1. Submit all chunks in parallel using .remote()
#   2. Collect all results with a single ray.get() call
#   3. Merge the per-chunk dicts into one global word-frequency dict
#   4. Print the top 5 most frequent words
#
# Do NOT call ray.get() inside the submission loop.

chunks = [
    "ray is a distributed computing framework for python",
    "ray tasks run in parallel across many worker processes",
    "ray actors hold state and run as long-lived worker processes",
    "ray provides a simple api for distributed python computing",
    "distributed computing with ray is fast and easy to use",
]

# Your solution here:
# @ray.remote
# def count_words(chunk: str) -> dict:
#     ...

# refs = [...]
# results = ray.get(refs)
# merged = {}
# ...
# top5 = sorted(merged.items(), key=lambda kv: kv[1], reverse=True)[:5]
# print(top5)


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `ray.init()` | Starts a local cluster; no args needed for local dev |
| `@ray.remote` | Converts a function into a distributed task |
| `.remote()` | Submits the task; returns `ObjectRef` immediately (non-blocking) |
| `ObjectRef` | A handle to a future result stored in the distributed object store |
| `ray.get(refs)` | Blocks until results are ready; accepts a single ref or a list |
| `ray.wait()` | Returns as soon as N tasks complete — enables streaming patterns |
| `num_cpus` | Resource hint to the scheduler; use fractional values for I/O-bound tasks |
| `.options()` | Override decorator-level resources at call time without redefining the function |

> **Tip:** ray.get() is a blocking call. For maximum parallelism, submit all .remote() calls first, collect ObjectRefs into a list, then call ray.get(refs) once at the end. Never interleave .remote() and ray.get() in a loop.

---
## What's next
**Day 2** → Actors and the Object Store — stateful distributed computing with `@ray.remote` classes, the Plasma object store, and `ActorPool` for throughput scaling.

Mark Day 1 complete in your [tracker](../index.html).


In [ ]:
# Shut down Ray cleanly at the end of the notebook
ray.shutdown()
print("Ray cluster shut down.")
